# Week 4 — Day 2

# Threshold Optimization Analysis

This notebook demonstrates threshold optimization for the predictive maintenance model.

## Objectives

- Load fused dataset
- Train LightGBM
- Optimize decision threshold
- Compare Precision, Recall, Accuracy and F1
- Visualize threshold performance

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------------------------
# Add src folder to Python path
# -------------------------------------------------

from pathlib import Path

# Directory containing this notebook
NOTEBOOK_DIR = Path.cwd()

# Project root (parent of notebooks)
PROJECT_ROOT = NOTEBOOK_DIR.parent

# src directory
SRC_PATH = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_PATH))

print("Current Working Directory:", NOTEBOOK_DIR)
print("Project Root:", PROJECT_ROOT)
print("SRC Path:", SRC_PATH)
print("SRC Exists:", SRC_PATH.exists())

print("Project Root :", PROJECT_ROOT)
print("SRC Path     :", SRC_PATH)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

from imblearn.over_sampling import SMOTE

import lightgbm as lgb

from external_data.data_fusion import (
    create_fused_dataset,
    get_fused_arrays
)

In [ ]:
PROJECT_ROOT = os.path.abspath("..")

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "ai4i2020.csv"
)

RESULT_DIR = os.path.join(
    PROJECT_ROOT,
    "src",
    "tuning_results"
)

PARAM_FILE = os.path.join(
    RESULT_DIR,
    "optuna_best_parameters.json"
)

TEST_SIZE = 0.20
RANDOM_STATE = 42

In [ ]:
print("="*60)
print("LOADING DATASET")
print("="*60)

fused_df = create_fused_dataset(DATA_PATH)

X, y, feature_names = get_fused_arrays(fused_df)

print("Dataset Shape :", X.shape)
print("Features      :", len(feature_names))
print("Failure Rate  :", round(y.mean()*100,2), "%")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Applying SMOTE...")

smote = SMOTE(random_state=RANDOM_STATE)

X_train, y_train = smote.fit_resample(
    X_train,
    y_train
)

print("Train Shape :", X_train.shape)
print("Test Shape  :", X_test.shape)

In [ ]:
print("=" * 60)
print("LOADING BEST PARAMETERS")
print("=" * 60)

if os.path.exists(PARAM_FILE):

    with open(PARAM_FILE, "r") as f:
        params = json.load(f)

    print("Optuna parameters loaded.")

else:

    params = {}
    print("Default parameters will be used.")

params.pop("objective", None)
params.pop("metric", None)
params.pop("boosting_type", None)
params.pop("random_state", None)
params.pop("class_weight", None)
params.pop("verbosity", None)

In [ ]:
print("=" * 60)
print("TRAINING LIGHTGBM")
print("=" * 60)

model = lgb.LGBMClassifier(
    objective="binary",
    metric="binary_logloss",
    boosting_type="gbdt",
    random_state=RANDOM_STATE,
    class_weight="balanced",
    verbosity=-1,
    **params
)

model.fit(
    X_train,
    y_train
)

probabilities = model.predict_proba(X_test)[:,1]

print("Model trained successfully.")

In [ ]:
thresholds = np.arange(0.05,1.00,0.05)

results = []

for threshold in thresholds:

    predictions = (probabilities >= threshold).astype(int)

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    results.append({
        "Threshold":threshold,
        "Precision":precision,
        "Recall":recall,
        "F1 Score":f1,
        "Accuracy":accuracy
    })

results = pd.DataFrame(results)

results.head()

In [ ]:
best_index = results["F1 Score"].idxmax()

best_row = results.iloc[best_index]

print("Best Threshold :", best_row["Threshold"])
print("Best F1 Score  :", round(best_row["F1 Score"],4))
print("Precision      :", round(best_row["Precision"],4))
print("Recall         :", round(best_row["Recall"],4))
print("Accuracy       :", round(best_row["Accuracy"],4))

In [ ]:
plt.figure(figsize=(12,7))

plt.plot(
    results["Threshold"],
    results["Precision"],
    label="Precision",
    linewidth=2
)

plt.plot(
    results["Threshold"],
    results["Recall"],
    label="Recall",
    linewidth=2
)

plt.plot(
    results["Threshold"],
    results["F1 Score"],
    label="F1 Score",
    linewidth=2
)

plt.plot(
    results["Threshold"],
    results["Accuracy"],
    label="Accuracy",
    linewidth=2
)

plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Threshold Optimization")
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
results.to_csv(
    os.path.join(
        RESULT_DIR,
        "threshold_results.csv"
    ),
    index=False
)

with open(
    os.path.join(
        RESULT_DIR,
        "best_threshold_report.txt"
    ),
    "w"
) as f:

    f.write("="*60 + "\n")
    f.write("THRESHOLD OPTIMIZATION REPORT\n")
    f.write("="*60 + "\n\n")

    f.write(f"Best Threshold : {best_row['Threshold']:.2f}\n")
    f.write(f"Precision      : {best_row['Precision']:.4f}\n")
    f.write(f"Recall         : {best_row['Recall']:.4f}\n")
    f.write(f"F1 Score       : {best_row['F1 Score']:.4f}\n")
    f.write(f"Accuracy       : {best_row['Accuracy']:.4f}\n")

print("Threshold optimization completed successfully.")